<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/main/7_clase_analisis_genoma_COVID19_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis del genoma de COVID-19 usando Biopython

**Nota para Google Colab:**  
- Ejecuta `!pip install biopython matplotlib` si aún no los tienes instalados.  
- Se requiere conexión a internet para descargar la secuencia desde el NCBI.  
- Reemplaza el correo electrónico por uno real.

### Obtener el genoma de COVID-19 desde el NCBI

**MN908947** – el genoma de COVID-19 utilizado aquí fue secuenciado a partir de una muestra de líquido de lavado broncoalveolar de un único paciente que era trabajador del mercado y que fue admitido en el Hospital Central de Wuhan el 26 de diciembre de 2019.

In [ ]:
from Bio import Entrez, SeqIO

# ¡Reemplaza por tu correo electrónico real!
Entrez.email = "tu_correo@ejemplo.com"

handle = Entrez.efetch(db="nucleotide", id="MN908947", rettype="gb", retmode="text")
recs = list(SeqIO.parse(handle, 'gb'))
handle.close()

In [ ]:
recs

In [ ]:
covid_dna = recs[0].seq

In [ ]:
covid_dna

In [ ]:
print(f'El genoma de COVID-19 consta de {len(covid_dna)} nucleótidos.')

In [ ]:
# Peso molecular
from Bio.SeqUtils import molecular_weight
molecular_weight(covid_dna)

In [ ]:
# Contenido de GC - un mayor contenido de GC implica una molécula más estable
# porque G y C forman tripletes de puentes de hidrógeno
from Bio.SeqUtils import gc_fraction

# Nota: en versiones recientes de Biopython se usa gc_fraction (devuelve 0-1)
# Multiplicamos por 100 para obtener el porcentaje
gc_fraction(covid_dna) * 100

### Distribución de nucleótidos en el genoma de COVID-19

In [ ]:
count_nucleotides = {
    'A': covid_dna.count('A'),
    'T': covid_dna.count('T'),
    'C': covid_dna.count('C'),
    'G': covid_dna.count('G')
}

In [ ]:
count_nucleotides

In [ ]:
import matplotlib.pyplot as plt

width = 0.5
plt.bar(count_nucleotides.keys(), count_nucleotides.values(), width, color=['b', 'r', 'm', 'c'])
plt.xlabel('Nucleótido')
plt.ylabel('Frecuencia')
plt.title('Frecuencia de nucleótidos')
plt.show()

### Transcripción y traducción del genoma

In [ ]:
# Transcripción (ADN → ARNm)
covid_mrna = covid_dna.transcribe()

# Traducción (ARNm → proteína)
covid_aa = covid_mrna.translate()
covid_aa

In [ ]:
from collections import Counter

common_amino = Counter(covid_aa)
common_amino

In [ ]:
# Eliminar el codón de parada (*)
del common_amino['*']

width = 0.5
plt.bar(common_amino.keys(), common_amino.values(), width, color=['b', 'r', 'm', 'c'])
plt.xlabel('Aminoácido')
plt.ylabel('Frecuencia')
plt.title('Frecuencia de aminoácidos en la secuencia proteica')
plt.show()

In [ ]:
print(f"El genoma de COVID-19 tiene {sum(common_amino.values())} aminoácidos")

La función `split()` divide la secuencia en cualquier codón de parada y mantiene las cadenas de aminoácidos separadas. Esto facilita el análisis posterior.

In [ ]:
proteins = covid_aa.split('*')

In [ ]:
proteins[:5]

In [ ]:
print(f'Tenemos {len(proteins)} cadenas de aminoácidos en el genoma de COVID-19')

Vale la pena mencionar que no todas las secuencias de aminoácidos son proteínas. Solo las secuencias con más de 20 aminoácidos codifican proteínas funcionales. Las secuencias cortas de aminoácidos son oligopéptidos y tienen otras funcionalidades. Aquí nos enfocaremos en las cadenas con más de 20 aminoácidos: las **proteínas**.

In [ ]:
# Filtrar solo las proteínas con más de 20 aminoácidos
proteins = [p for p in proteins if len(p) >= 20]

print(f'Tenemos {len(proteins)} proteínas con más de 20 aminoácidos en el genoma de COVID-19')

In [ ]:
top_5_proteins = sorted(proteins, key=len)

In [ ]:
top_5_proteins[-1]

In [ ]:
len(top_5_proteins[-1])

Guardamos esta proteína en un archivo para análisis posteriores.

In [ ]:
with open("protein_seq.fasta", "w") as file:
    file.write(f">covid protein\n{top_5_proteins[-1]}")

print("Archivo 'protein_seq.fasta' guardado exitosamente.")

## Resumen de hallazgos

- Longitud de la secuencia: **29,903** pares de bases
- Contenido de GC: **≈ 37.97%**
- Alta cantidad de Leucina (L) y Serina (S)
- **≈ 409** proteínas con más de 20 aminoácidos
- La proteína más grande tiene una longitud de **2,701** aminoácidos